<a href="https://colab.research.google.com/github/sun-mengwei/dtb-colab-experiments/blob/codex%2Fgame-dynamics-dtb/DTB_Game_Ver2/cournot_10d_nonpotential_mlp_dtb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cournot game: ordinary neural DTB map update

The notebook reuses `ResidualMLPMap`, `game_dtb_basis_matrices`, and `map_at` from `run_game_dtb.py`; `count_trainable` from `network.py`; and `device`, `flat_params`, and `jform_solve` from `dtb.py`.

Plots are generated by the reusable `game_visualization.py` utility. It infers the state dimension and supports NumPy arrays or PyTorch tensors.

The notebook follows four blocks: **define the game velocity**, **initialize an MLP tangent representation**, **update the pushforward map**, and **save diagnostics and plots**.

Use a fixed sample of labels \(z_j\) and initialize \(X_0(z)=z\). The MLP \(f_{\theta_0}:\mathbb{R}^{d}\to\mathbb{R}^{d}\) supplies the tangent basis, with `d = DIM`; it is separate from the evolving map \(X_k\). Its parameters stay fixed in this map-only experiment. A reproducible random subset $S$ of `BASIS_SIZE` parameter directions defines the tangent representation and remains fixed for every time step.

At each step, evaluate the parameter Jacobian at the **current physical points**:

\[
x_{k,j}=X_k(z_j),\qquad J_{k,j}^{S}=\partial_{\theta_S} f_{\theta_0}(x_{k,j}),\qquad
g_{k,j}=b(x_{k,j}).
\]

Solve the tangent projection and update the map:

\[
\alpha_k=\arg\min_{\alpha\in\mathbb{R}^{|S|}}\frac1N\sum_j\|J_{k,j}^{S}\alpha-g_{k,j}\|^2,
\qquad u_k(x)=\partial_{\theta_S} f_{\theta_0}(x)\alpha_k,
\qquad X_{k+1}(z)=X_k(z)+h\,u_k(X_k(z)).
\]

The normal equations are \(G_k\alpha_k=P_k\), with \(G_k=N^{-1}\sum_j(J_{k,j}^{S})^{\mathsf T}J_{k,j}^{S}\) and \(P_k=N^{-1}\sum_j(J_{k,j}^{S})^{\mathsf T}g_{k,j}\). The code solves the equivalent stacked least-squares problem with a truncated SVD. The common \(1/N\) factor does not change the solution. The Jacobian and its SVD are rebuilt each step because the physical points move.

This is the fixed-sample, deterministic variant: labels are drawn once, the target velocity is just \(b(x)\), and every map step uses the tangent projection. All diagnostics use the actual DTB trajectory.

In [ ]:
# BLOCK 1 — Define the d-player velocity field and experiment controls.
from pathlib import Path
from itertools import combinations
import csv
import json
import time
from datetime import datetime, timezone
import subprocess
import sys
import numpy as np
import torch
from torch.func import jvp

# Use sibling repository modules locally; fetch the same branch in Colab.
module_files = ('run_game_dtb.py', 'network.py', 'dtb.py', 'game_visualization.py')
EXPERIMENT_DIR = next((folder for folder in (Path.cwd(), Path.cwd() / 'DTB_Game_Ver2')
                       if all((folder / name).is_file() for name in module_files)), None)
if EXPERIMENT_DIR is None:
    if not Path('/content').is_dir():
        raise FileNotFoundError('Run this notebook from the repository root or DTB_Game_Ver2 directory.')
    repo = Path('/content/dtb-colab-experiments')
    if not (repo / '.git').exists():
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'codex/game-dynamics-dtb',
                        'https://github.com/sun-mengwei/dtb-colab-experiments.git', str(repo)], check=True)
    else:
        subprocess.run(['git', '-C', str(repo), 'checkout', 'codex/game-dynamics-dtb'], check=True)
        subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
    EXPERIMENT_DIR = repo / 'DTB_Game_Ver2'
sys.path.insert(0, str(EXPERIMENT_DIR.resolve()))

from run_game_dtb import ResidualMLPMap, game_dtb_basis_matrices, map_at
from network import count_trainable
from dtb import device, flat_params, jform_solve
# Refresh the plotting utility when rerunning in an existing Colab session.
from importlib import reload
import game_visualization
save_game_visualizations = reload(game_visualization).save_game_visualizations

# Set this to False to display the run without creating any output folder.
SAVE_RUN = True

SEED = 2026
DIM = 3
N_PARTICLES = 384
T_FINAL, H = 1.0, 0.01
N_STEPS = round(T_FINAL / H)
NN_CHOICE = 'mlp'
NN_ACTIVATION = 'tanh'
MLP_WIDTH, MLP_DEPTH = 24, 2
BASIS_SIZE = 128  # Number of randomly selected MLP parameter directions.
COORDINATE_PAIR_SEED = SEED  # Reproducible plot-plane selection when DIM > 5.
JACOBIAN_CHUNK = 64
SVD_RTOL = 1e-3
INITIAL_TOTAL = 1.0
COURNOT_B, COURNOT_MU = 1.0, 2.0
DEVICE = device()
DTYPE = torch.float32
assert N_STEPS > 0 and H > 0 and np.isclose(N_STEPS * H, T_FINAL)
torch.manual_seed(SEED)
if DEVICE.type == 'cuda':
    torch.cuda.manual_seed_all(SEED)

def cournot_velocity(x):
    """b_i(x) = 2 B [max(mu r_i (1-r_i), 0) - x_i], r_i = sum_{j != i} x_j."""
    opponents = x.sum(dim=-1, keepdim=True) - x
    best_response = (COURNOT_MU * opponents * (1.0 - opponents)).clamp_min(0.0)
    return 2.0 * COURNOT_B * (best_response - x)

# Analytic reference points for read-only equilibrium-distance diagnostics.
# This is a known subset of equilibria, not an exhaustive classification.
known_points = [torch.zeros(DIM, device=DEVICE, dtype=DTYPE)]
known_names = ['origin']
q = (COURNOT_MU * (DIM - 1) - 1.0) / (COURNOT_MU * (DIM - 1) ** 2)
if q > 0:
    known_points.append(torch.full((DIM,), q, device=DEVICE, dtype=DTYPE))
    known_names.append('symmetric')
if COURNOT_MU == 2.0:
    for first, second in combinations(range(DIM), 2):
        point = torch.zeros(DIM, device=DEVICE, dtype=DTYPE)
        point[first] = point[second] = 0.5
        known_points.append(point)
        known_names.append(f'pair({first + 1},{second + 1})')
known_equilibria = torch.stack(known_points)
assert torch.linalg.vector_norm(cournot_velocity(known_equilibria), dim=1).max() < 2e-5
print({'device': str(DEVICE), 'players': DIM, 'particles': N_PARTICLES,
       'steps': N_STEPS, 'final_time': T_FINAL})

In [ ]:
# BLOCK 2 — Initialize X_0(z)=z and the MLP used to generate the tangent plane.
# Draw labels once. These same labels are used throughout the experiment.
weights = -torch.rand(N_PARTICLES, DIM, device=DEVICE, dtype=DTYPE).clamp_min(1e-7).log()
labels = INITIAL_TOTAL * weights / weights.sum(dim=1, keepdim=True)
x_0 = labels.clone()

# Reuse the existing MLP architecture; X_0 remains the identity independently.
mlp = ResidualMLPMap(
    dim=DIM, width=MLP_WIDTH, depth=MLP_DEPTH, activation=NN_ACTIVATION,
    dtype=DTYPE, zero_init_output=False,
).net.to(DEVICE)
N_PARAMETERS = count_trainable(mlp)
theta_0, structure, _ = flat_params(mlp)
parameter_names = [name for name, _ in structure]
parameter_shapes = [shape for _, shape in structure]
# Select one reproducible random tangent subset and keep it fixed across time.
if not isinstance(BASIS_SIZE, int) or not 1 <= BASIS_SIZE <= N_PARAMETERS:
    raise ValueError(f'BASIS_SIZE must be an integer in [1, {N_PARAMETERS}]')
basis_generator = torch.Generator(device='cpu')
basis_generator.manual_seed(SEED)
selected = torch.randperm(N_PARAMETERS, generator=basis_generator)[:BASIS_SIZE].sort().values.to(DEVICE)
mlp.requires_grad_(False)

@torch.no_grad()
def evaluate_map(initial_points, coefficients, theta):
    """Replay the composed map X_k on any input labels, using stored alpha_0,...,alpha_{k-1}."""
    current = initial_points.to(device=DEVICE, dtype=DTYPE).clone()
    for alpha_selected in coefficients:
        alpha_selected = alpha_selected.to(device=DEVICE, dtype=DTYPE)
        full_direction = torch.zeros_like(theta).index_copy(0, selected, alpha_selected)
        _, velocity = jvp(lambda parameters: map_at(parameters, current, mlp, structure),
                          (theta,), (full_direction,))
        current = current + H * velocity
    return current

print({'MLP': str(mlp), 'MLP_parameters': N_PARAMETERS,
       'basis_size': BASIS_SIZE,
       'stacked_J_shape': (N_PARTICLES * DIM, BASIS_SIZE)})

In [ ]:
# BLOCK 3 — Update the map using its tangent velocity and record each step.
def synchronize():
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()

def rms_vector(values):
    return values.square().sum(dim=-1).mean().sqrt()

def state_diagnostics(points, step):
    nearest = torch.cdist(points, known_equilibria).min(dim=1).values
    return {
        'step': step, 'time': step * H,
        'game_drift_rms': float(rms_vector(cournot_velocity(points))),
        'median_known_distance': float(nearest.median()),
        'p90_known_distance': float(torch.quantile(nearest, 0.9)),
        'minimum_coordinate': float(points.min()),
        'negative_coordinate_fraction': float((points < 0).float().mean()),
    }

def solve_projection(A, target):
    """Use the shared DTB solver, then measure the projection residual."""
    alpha = jform_solve(A, target, rtol=SVD_RTOL, method='svd_gpu')
    residual = A @ alpha - target
    return alpha, {
        'projection_residual': float(residual.norm() / target.norm().clamp_min(1e-30)),
        'alpha_norm': float(alpha.norm()),
    }

@torch.no_grad()
def run_dtb(initial_points, theta):
    current = initial_points.clone()
    trajectory = [current.cpu()]
    coefficients = []
    states = [state_diagnostics(current, 0)]
    projections = []
    synchronize()
    run_start = time.perf_counter()

    for step in range(N_STEPS):
        # x_{k,j} = X_k(z_j); target g_{k,j} = b(x_{k,j}).
        target = cournot_velocity(current)
        synchronize()
        basis_start = time.perf_counter()
        _, J, A = game_dtb_basis_matrices(
            theta, selected, current, mlp, structure, chunk=JACOBIAN_CHUNK
        )
        synchronize()
        solve_start = time.perf_counter()

        # This solves G_k alpha_k = P_k through its stacked least-squares form.
        alpha, info = solve_projection(A, target.reshape(-1))
        synchronize()
        update_start = time.perf_counter()

        # u_k(X_k(z)) = J_theta(X_k(z)) alpha_k; update the map at the fixed labels.
        projected_velocity = (A @ alpha).reshape_as(current)
        current = current + H * projected_velocity
        if not torch.isfinite(current).all():
            raise FloatingPointError(f'Non-finite DTB state at step {step + 1}')
        synchronize()
        update_end = time.perf_counter()

        coefficients.append(alpha.cpu())
        trajectory.append(current.cpu())
        states.append(state_diagnostics(current, step + 1))
        projections.append({
            'step': step, 'time': step * H, **info,
            'projected_drift_rms': float(rms_vector(projected_velocity)),
            'basis_seconds': solve_start - basis_start,
            'solve_seconds': update_start - solve_start,
            'update_seconds': update_end - update_start,
        })
        if (step + 1) % max(1, N_STEPS // 10) == 0:
            print(f"step {step + 1}/{N_STEPS}: projection residual={info['projection_residual']:.3e}, "
                  f"endpoint drift RMS={states[-1]['game_drift_rms']:.3e}")

    synchronize()
    return {'trajectory': torch.stack(trajectory),
            'coefficients': torch.stack(coefficients),
            'states': states, 'projections': projections,
            'wall_seconds': time.perf_counter() - run_start}

RUN_STARTED_UTC = datetime.now(timezone.utc)
result = run_dtb(x_0, theta_0)
trajectory = result['trajectory']
dtb_endpoints = trajectory[-1]
print({'DTB_wall_seconds': result['wall_seconds'], **result['states'][-1]})

# The global map is available on new labels as well as the stored particles:
# new_endpoints = evaluate_map(new_labels, result['coefficients'], theta_0)
# For X_k, pass result['coefficients'][:k] instead.

In [ ]:
# BLOCK 4 — Optionally save this run; always display its figures and mathematical metrics.
run_stamp = RUN_STARTED_UTC.strftime('%Y%m%dT%H%M%S-%fZ')
RUN_FOLDER_NAME = (
    f'cournot-{DIM}d-nonpotential__nn-{NN_CHOICE}-{NN_ACTIVATION}'
    f'_w{MLP_WIDTH}-d{MLP_DEPTH}-p{N_PARAMETERS}'
    f'__basis-{BASIS_SIZE}__seed-{SEED}__{run_stamp}'
)
OUTPUT_DIR = EXPERIMENT_DIR / 'saved_runs' / RUN_FOLDER_NAME if SAVE_RUN else None

configuration = {
    'save_run': SAVE_RUN, 'run_started_utc': RUN_STARTED_UTC.isoformat(),
    'run_folder': RUN_FOLDER_NAME, 'seed': SEED, 'dim': DIM,
    'game': 'cournot_nonpotential', 'particles': N_PARTICLES, 'steps': N_STEPS,
    'h': H, 'final_time': T_FINAL, 'nn_choice': NN_CHOICE,
    'nn_activation': NN_ACTIVATION, 'mlp_width': MLP_WIDTH, 'mlp_depth': MLP_DEPTH,
    'nn_parameters': N_PARAMETERS, 'basis_size': BASIS_SIZE,
    'coordinate_pair_seed': COORDINATE_PAIR_SEED,
    'svd_rtol': SVD_RTOL, 'initial_total': INITIAL_TOTAL,
    'cournot_b': COURNOT_B, 'cournot_mu': COURNOT_MU,
    'jacobian_chunk': JACOBIAN_CHUNK, 'dtype': str(DTYPE), 'device': str(DEVICE),
    'wall_seconds': result['wall_seconds'],
}

if SAVE_RUN:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=False)

    def save_table(filename, rows):
        with (OUTPUT_DIR / filename).open('w', newline='') as handle:
            writer = csv.DictWriter(handle, fieldnames=list(rows[0]), lineterminator='\n')
            writer.writeheader()
            writer.writerows(rows)

    save_table('state_diagnostics.csv', result['states'])
    save_table('projection_diagnostics.csv', result['projections'])

# Up to 5D, show (x1,x2) and (x3,x4) when available. Above 5D,
# choose three distinct coordinate pairs reproducibly from COORDINATE_PAIR_SEED.
# Explicit pairs still override this default: coordinate_pairs=[(1, 3), (2, 5)].
visualizations = save_game_visualizations(
    trajectory, h=H, output_dir=OUTPUT_DIR, save_files=SAVE_RUN,
    equilibria=known_equilibria, coordinate_seed=COORDINATE_PAIR_SEED,
    state_diagnostics=result['states'], projection_diagnostics=result['projections'],
)
configuration['coordinate_pairs'] = [list(pair) for pair in visualizations['coordinate_pairs']]

if SAVE_RUN:
    (OUTPUT_DIR / 'configuration.json').write_text(json.dumps(configuration, indent=2) + '\n')
    torch.save({
        'configuration': configuration, 'theta_0': theta_0.cpu(),
        'parameter_names': parameter_names, 'parameter_shapes': parameter_shapes,
        'selected_parameter_indices': selected.cpu(),
        'labels': labels.cpu(), 'trajectory': trajectory,
        'coefficients': result['coefficients'],
        'known_equilibria': known_equilibria.cpu(), 'known_names': known_names,
    }, OUTPUT_DIR / 'dtb_run.pt')
    pair_text = ', '.join(f'(x_{first}, x_{second})'
                          for first, second in visualizations['coordinate_pairs'])
    readme = f'''# Saved Cournot DTB run

- Game: {DIM}D non-potential Cournot game
- Neural network: {NN_CHOICE}, activation={NN_ACTIVATION}, width={MLP_WIDTH}, depth={MLP_DEPTH}
- Trainable MLP parameters: {N_PARAMETERS}
- Selected tangent basis size: {BASIS_SIZE}
- Displayed coordinate pairs: {pair_text}
- Seed: {SEED}
- Started (UTC): {RUN_STARTED_UTC.isoformat()}

The folder contains the coordinate projections, tangent-projection diagnostics,
mathematical metric table, full diagnostic histories, configuration, and saved DTB state.
'''
    (OUTPUT_DIR / 'README.md').write_text(readme)
    print('Saved run inside the Git repository at:', OUTPUT_DIR.resolve())
else:
    print('SAVE_RUN=False: displayed results without creating an output folder.')


The initial label distribution is sampled once on the simplex. Each subsequent state is produced by the ordinary tangent-bundle map update. The MLP parameters remain \(\theta_0\), while the selected Jacobian \(J_{k,j}^{S}=\partial_{\theta_S}f_{\theta_0}(X_k(z_j))\) changes with the physical points. The subset \(S\) is sampled once from all MLP parameter coordinates using the seeded generator; set `BASIS_SIZE` to control \(|S|\). There is no optimizer step on the MLP.

`dtb_run.pt` stores the initial labels, frozen MLP parameters, selected parameter indices, reduced tangent coefficients, and the full particle trajectory. Together with the saved configuration, the coefficients specify the composed global map. `evaluate_map` applies that map to arbitrary input labels. During the run, the fixed projection particles also serve as diagnostic particles.

`state_diagnostics.csv` includes the initial state and every updated state. `projection_diagnostics.csv` records each tangent solve at its input time \(t_k\). Endpoint residuals and distances are measured at \(T_{\mathrm{FINAL}}\); particle coordinates are kept exactly as produced by DTB. A small projected velocity alone does not establish equilibrium when the game-velocity residual is large. Up to 5D, coordinate snapshots show $(x_1,x_2)$ and $(x_3,x_4)$ when available. Above 5D, they show three seeded random coordinate planes. Every row keeps the same axis limits across time and includes projected reference equilibria and short trails for fixed particles. The marker legend reports the number of full-state reference equilibria and the number of distinct locations in that plane; multiplicity labels show how many full equilibria collapsed onto each repeated location. These views can overlap in hidden coordinates; equilibrium distances use the full state.

Increase `T_FINAL` to study longer-time behavior using the same map update. The shared solver returns tangent coefficients; the notebook records their norm and the projection residual without performing a second SVD for rank diagnostics. Both the explicit parameter Jacobian and its SVD are recomputed every step, so `N_PARTICLES` and `MLP_WIDTH` affect runtime and memory.

The visualization utility generates the coordinate-plane snapshots and a mathematical metric table:

- `coordinate_snapshots.png`: the default low-dimensional planes or three seeded random coordinate pairs above 5D, shown across time. Axes remain fixed within each row. Red markers show projected reference equilibria, marker labels show projection multiplicities, and short trails follow fixed particles.
- `projection_metrics.png`: time series of the relative tangent-projection error \(r_k\) and selected coefficient norm \(\lVert\alpha_k\rVert_2\).
- `dtb_metrics.md`: mathematical definitions and the first/last recorded values of the game and tangent RMS velocities, relative projection residual, equilibrium-distance statistics, minimum coordinate, negative-coordinate fraction, coefficient norm, and operation timings. The table is also rendered directly in the notebook.

State metrics include $x_K$ at $T$. Projection metrics describe the input state $x_k$ used for each solve, ending at $T-h$. The tables label these times separately. The diagnostic CSV files retain every recorded time step.

For a 3D, 5D, or 6D experiment, the same visualization call works with the corresponding trajectory and reference points. Change `coordinate_pairs` or `snapshot_steps` only when desired; no plotting implementation needs to be copied into the notebook.

Set `SAVE_RUN` at the beginning of Block 1. When it is `True`, every execution of Block 3 receives a unique UTC timestamp and Block 4 creates a new directory under `DTB_Game_Ver2/saved_runs/`. Its name records the game dimension, neural-network choice, activation, width, depth, total MLP parameter count, tangent-basis size, and seed. Because this directory is inside the checked-out repository, it becomes a GitHub folder when committed and pushed. When `SAVE_RUN=False`, both figures and the mathematical table are displayed without writing files.

For this Cournot reference set with $\mu=2$, the selected reference subset in an 8D game has
$1+1+\binom{8}{2}=30$ equilibrium points, while the 5D subset has
$1+1+\binom{5}{2}=12$. A two-coordinate projection collapses either full set
onto five visible locations. The central symmetric location still changes with
$d$ through $q_d=[\mu(d-1)-1]/[\mu(d-1)^2]$: $q_8=13/98$ and $q_5=7/32$.
The legend and marker multiplicities distinguish the full reference set from its projection.
